# 14장. 신경망의 해석

학습한 신경망의 예측을 입력의 어느 부분이 정했는지를 재는 방법을 만든다. 기울기, 기울기×입력, 적분 기울기, 가림 실험을 9장의 합성곱 신경망에 쓰고, 그 결과를 믿을 수 있는지 확인하는 실험을 한다. 11장의 어텐션 모델에서는 어텐션 가중치를 같은 방법으로 확인한다.

**준비:** 9장의 파일 `train-images-idx3-ubyte.gz`, `train-labels-idx1-ubyte.gz`(https://github.com/zalandoresearch/fashion-mnist)와 11장의 파일 `ratings_train.txt`(https://github.com/e9t/nsmc)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 두 자료의 시험용 파일은 16장에서 쓴다. 끝까지 실행하는 데 6분쯤 걸린다.

## 14.1 입력의 기울기로 예측에 영향을 준 픽셀 찾기

In [ ]:
import copy, gzip, math, time
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import rankdata
from torch import nn

torch.set_num_threads(2)

def read_idx(path, offset):
    """idx 형식의 파일을 읽는다. 앞의 offset바이트는 머리말이다."""
    with gzip.open(path) as f:
        return np.frombuffer(f.read(), np.uint8, offset=offset)

images = read_idx("train-images-idx3-ubyte.gz", 16).reshape(-1, 28, 28)
labels = read_idx("train-labels-idx1-ubyte.gz", 8).astype(np.int64)
names = ["티셔츠", "바지", "스웨터", "드레스", "코트",
         "샌들", "셔츠", "운동화", "가방", "앵클부츠"]
order = np.random.default_rng(0).permutation(60000)           # 9장과 같은 나누기
train_rows, valid_rows = order[:6000], order[50000:]

def tensors(rows, mean=None, std=None):
    """고른 이미지를 (장수, 1, 28, 28) 모양의 실수 텐서로 바꾸고 표준화한다."""
    x = torch.tensor(images[rows] / 255, dtype=torch.float32)[:, None]
    if mean is None:
        mean, std = x.mean().item(), x.std().item()
    return (x - mean) / std, torch.tensor(labels[rows]), mean, std

X_tr, y_tr, pixel_mean, pixel_std = tensors(train_rows)
X_va, y_va, _, _ = tensors(valid_rows, pixel_mean, pixel_std)
blank = (0 - pixel_mean) / pixel_std               # 검은 바탕(0)이 표준화된 값
white = (1 - pixel_mean) / pixel_std               # 가장 큰 값(255)이 표준화된 값
print(X_tr.shape, X_va.shape, round(blank, 4), round(white, 4))

In [ ]:
def accuracy(net, x, y):
    """정확도(%). 1,000장씩 나눠 예측한다."""
    net.eval()
    with torch.no_grad():
        guess = torch.cat([net(x[k:k + 1000]).argmax(dim=1)
                           for k in range(0, len(x), 1000)])
    return (guess == y).float().mean().item() * 100

def fit(net, x, y, seed, epochs=10, lr=0.001):
    """9장의 학습 방법: Adam, 배치 크기 64, 10에폭."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    shuffler = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train()
        mixed = torch.randperm(len(x), generator=shuffler)
        for start in range(0, len(x), 64):
            rows = mixed[start:start + 64]
            optimizer.zero_grad()
            F.cross_entropy(net(x[rows]), y[rows]).backward()
            optimizer.step()
    return net

def make_cnn():
    """9.3절의 모델: 합성곱(16) - 풀링 - 합성곱(32) - 풀링 - 선형."""
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, 10))

def train_five(x, y):
    """시드 0~4로 학습시킨 모델 5개."""
    made = []
    for seed in range(5):
        torch.manual_seed(seed)
        made.append(fit(make_cnn(), x, y, seed))
    return made

lap = time.perf_counter()
nets = train_five(X_tr, y_tr)
took = {"합성곱 신경망 5개": time.perf_counter() - lap}
net = nets[0]                                       # 이 장에서 주로 쓰는 모델
print(sum(p.numel() for p in net.parameters()),
      [round(accuracy(one, X_va, y_va), 2) for one in nets])

In [ ]:
def f(x):
    """입력 2개, ReLU 유닛 2개를 더한 작은 함수."""
    return max(2 * x[0] - x[1], 0) + max(x[0] + x[1] - 1, 0)

def slope(x):
    """f의 기울기. 켜진 유닛의 가중치를 더한다."""
    first, second = 2 * x[0] - x[1] > 0, x[0] + x[1] - 1 > 0
    return np.array([2.0, -1.0]) * first + np.array([1.0, 1.0]) * second

point, start = np.array([1.0, 0.5]), np.array([0.0, 0.0])
print(f(point), f(start), slope(point))
print(slope(point) * (point - start), (slope(point) * (point - start)).sum())
for steps in [3, 30, 300]:
    ratios = (np.arange(steps) + 0.5) / steps        # 기준 입력에서 입력까지의 비율
    mean_slope = np.mean([slope(start + a * (point - start)) for a in ratios], axis=0)
    integrated = mean_slope * (point - start)
    print(steps, mean_slope.round(4), integrated.round(4), integrated.sum().round(4))

In [ ]:
def grad_map(net, x, target):
    """이미지마다 target 클래스의 점수를 입력으로 미분한 값. x와 모양이 같다."""
    x = x.clone().requires_grad_(True)
    net.eval()
    net(x)[torch.arange(len(x)), target].sum().backward()
    return x.grad.detach()

N = 1000                                            # 해석할 검증 이미지의 수
x, y = X_va[:N], y_va[:N]
with torch.no_grad():
    scores = net(x)
pred = scores.argmax(dim=1)                         # 모델이 고른 클래스
G = grad_map(net, x, pred)
print(G.shape, names[pred[0]], names[y[0]], round(scores[0, pred[0]].item(), 4))
print(round(G[0, 0, 14, 14].item(), 6), round(G.abs().max().item(), 4))

exact = copy.deepcopy(net).double()                 # 64비트 실수로 바꿔 비교한다
x0, nudge = x[:1].double(), torch.zeros(1, 1, 28, 28, dtype=torch.float64)
nudge[0, 0, 14, 14] = 1e-4
with torch.no_grad():
    up, down = exact(x0 + nudge)[0, pred[0]], exact(x0 - nudge)[0, pred[0]]
by_autograd = grad_map(exact, x0, pred[:1])[0, 0, 14, 14].item()
print((up - down).item() / 2e-4, by_autograd)

In [ ]:
def ig_map(net, x, target, steps=64):
    """적분 기울기. 검은 이미지에서 x까지의 기울기를 평균해 (x - 기준)을 곱한다."""
    base, total = torch.full_like(x, blank), torch.zeros_like(x)
    for a in (torch.arange(steps) + 0.5) / steps:
        total += grad_map(net, base + a * (x - base), target)
    return total / steps * (x - base)

rows = torch.arange(200)
with torch.no_grad():
    dark = torch.full_like(x[:200], blank)
    lift = (net(x[:200]) - net(dark))[rows, pred[:200]]   # 검은 이미지와의 점수 차이
print("걸음 수   합과 점수 차이의 가장 큰 어긋남   (점수 차이의 중앙값 "
      f"{lift.median().item():.3f})")
for steps in [16, 64, 128]:
    total = ig_map(net, x[:200], pred[:200], steps).sum(dim=(1, 2, 3))
    print(f"{steps:>5}     {(total - lift).abs().max().item():.4f}")

lap = time.perf_counter()
GI, IG = G * (x - blank), ig_map(net, x, pred)      # 기울기×입력, 적분 기울기
took["적분 기울기 1,000장"] = time.perf_counter() - lap
thing = (x > blank + 1e-6)                          # 값이 0보다 큰 픽셀(물체)
print(round(thing.float().mean().item(), 4),
      round(thing.flatten(1).float().mean(dim=1).median().item(), 4))
for name, picture in [("기울기", G), ("기울기×입력", GI), ("적분 기울기", IG)]:
    size = picture.abs()
    outside = (size * ~thing).sum(dim=(1, 2, 3)) / size.sum(dim=(1, 2, 3))
    print(name, round(outside.median().item(), 4))  # 크기의 합 가운데 바탕에 있는 비율

## 14.2 기울기 해석의 검증: 가중치를 무작위로 바꾼 모델과 비교하기

In [ ]:
torch.manual_seed(100)
untrained = make_cnn()                              # 한 번도 학습시키지 않은 모델
relaid = copy.deepcopy(net)                         # 학습한 모델의 마지막 층만 새로
torch.manual_seed(101)
relaid[-1].reset_parameters()
print(round(accuracy(untrained, X_va, y_va), 2), round(accuracy(relaid, X_va, y_va), 2))

def three_maps(model, steps=64):
    """세 방법의 그림. 부호를 떼고 크기만 남긴다."""
    slopes = grad_map(model, x, pred)
    return {"기울기": slopes.abs(), "기울기×입력": (slopes * (x - blank)).abs(),
            "적분 기울기": ig_map(model, x, pred, steps).abs()}

def rank_match(a, b, where=None):
    """이미지마다 두 그림의 순위 상관(스피어만)을 구해 중앙값을 낸다.
    where가 있으면 그 픽셀만 쓴다."""
    found = []
    for k in range(len(a)):
        keep = slice(None) if where is None else where[k].flatten()
        ra = rankdata(a[k].flatten()[keep].numpy())        # 값이 같으면 순위의 평균을 준다
        rb = rankdata(b[k].flatten()[keep].numpy())
        ra, rb = ra - ra.mean(), rb - rb.mean()
        found.append((ra * rb).sum() / (np.linalg.norm(ra) * np.linalg.norm(rb) + 1e-12))
    return float(np.median(found))

lap = time.perf_counter()
maps = {"학습한 모델": three_maps(net), "학습하지 않은 모델": three_maps(untrained),
        "마지막 층만 새로": three_maps(relaid)}
took["무작위 모델의 그림"] = time.perf_counter() - lap
bright = x - blank                                  # 이미지 자체: 픽셀의 값
match = {}
print("학습한 모델의 그림과의 순위 상관 (이미지 1,000장의 중앙값)")
print("                 모든 픽셀 784개               물체 픽셀만")
print("방법            무작위  마지막 층  이미지    무작위  마지막 층  이미지")
for name in maps["학습한 모델"]:
    mine = maps["학습한 모델"][name]
    others = [maps["학습하지 않은 모델"][name], maps["마지막 층만 새로"][name], bright]
    match[name] = [rank_match(mine, other) for other in others]
    match[name] += [rank_match(mine, other, thing) for other in others]
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (15 - len(name) - wide)
          + "".join(f"{v:>7.3f}   " for v in match[name][:3]) + " "
          + "".join(f"{v:>7.3f}   " for v in match[name][3:]).rstrip())

## 14.3 입력의 일부를 가리는 실험

In [ ]:
def occ_map(net, x, target, size=4):
    """size×size 칸을 하나씩 검게 가리고 target 점수가 내려간 정도를 그 칸에 적는다."""
    net.eval()
    drop, rows = torch.zeros_like(x), torch.arange(len(x))
    with torch.no_grad():
        full = net(x)[rows, target]
        for r in range(0, 28, size):
            for c in range(0, 28, size):
                covered = x.clone()
                covered[:, :, r:r + size, c:c + size] = blank
                gone = full - net(covered)[rows, target]
                drop[:, :, r:r + size, c:c + size] = gone[:, None, None, None]
    return drop

lap = time.perf_counter()
for name, model in [("학습한 모델", net), ("학습하지 않은 모델", untrained),
                    ("마지막 층만 새로", relaid)]:
    maps[name]["가림"] = occ_map(model, x, pred).abs()
took["가림 3,000장"] = time.perf_counter() - lap
OC = occ_map(net, x, pred)                          # 부호가 있는 값
print(OC.shape, 28 // 4 * (28 // 4), round(OC[0].max().item(), 4),
      round((OC > 0).float().mean().item(), 4))
mine = maps["학습한 모델"]["가림"]
others = [maps["학습하지 않은 모델"]["가림"], maps["마지막 층만 새로"]["가림"], bright]
match["가림"] = [rank_match(mine, other) for other in others]
match["가림"] += [rank_match(mine, other, thing) for other in others]
print("가림           " + "".join(f"{v:>7.3f}   " for v in match["가림"][:3]) + " "
      + "".join(f"{v:>7.3f}   " for v in match["가림"][3:]).rstrip())

In [ ]:
def keeps(picture, share):
    """picture의 값이 큰 픽셀부터 share만큼 검게 지운 뒤, 예측이 그대로인 비율(%)."""
    k = int(784 * share)
    top = picture.flatten(1).argsort(dim=1, descending=True)[:, :k]
    erased = x.clone().flatten(1)
    erased.scatter_(1, top, blank)
    with torch.no_grad():
        after = net(erased.reshape(N, 1, 28, 28)).argmax(dim=1)
    return (after == pred).float().mean().item() * 100

picker = torch.Generator().manual_seed(0)
by_chance = torch.rand(N, 1, 28, 28, generator=picker) * thing   # 물체 픽셀만 무작위
orders = {"가림": maps["학습한 모델"]["가림"],
          "적분 기울기": maps["학습한 모델"]["적분 기울기"],
          "기울기×입력": maps["학습한 모델"]["기울기×입력"],
          "기울기": maps["학습한 모델"]["기울기"],
          "값이 큰 픽셀부터": bright,
          "물체 픽셀 가운데 무작위": by_chance,
          "무작위 모델의 적분 기울기": maps["학습하지 않은 모델"]["적분 기울기"]}
SHARES = [0.02, 0.05, 0.1, 0.2]
kept = {name: [keeps(picture, share) for share in SHARES]
        for name, picture in orders.items()}
print("지운 픽셀의 비율            2%      5%     10%     20%   (예측이 그대로인 %)")
for name, row in kept.items():
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (26 - len(name) - wide) + "".join(f"{v:>8.1f}" for v in row))

In [ ]:
METHODS = ["기울기", "기울기×입력", "적분 기울기", "가림"]
between = np.array([[rank_match(maps["학습한 모델"][a], maps["학습한 모델"][b], thing)
                     for b in METHODS] for a in METHODS])
print("물체 픽셀에서 두 방법의 순위 상관 (학습한 모델, 이미지 1,000장의 중앙값)")
print(between.round(3))

lap = time.perf_counter()
by_seed = [maps["학습한 모델"]]                     # 시드 0은 이미 계산했다
for other in nets[1:]:
    made = three_maps(other)
    made["가림"] = occ_map(other, x, pred).abs()
    by_seed.append(made)
took["시드 4개의 그림"] = time.perf_counter() - lap
print("같은 이미지, 같은 클래스에 두 시드가 그린 그림의 순위 상관 (물체 픽셀)")
across = {}
for name in METHODS:
    pairs = [rank_match(by_seed[a][name], by_seed[b][name], thing)
             for a in range(5) for b in range(a + 1, 5)]
    across[name] = (np.median(pairs), min(pairs), max(pairs))
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (15 - len(name) - wide)
          + f"{across[name][0]:.3f} ({across[name][1]:.3f}~{across[name][2]:.3f})")

In [ ]:
def top_cell(picture):
    """가림 그림에서 값이 가장 큰 칸의 번호(0~48). 7행 7열을 한 줄로 센다."""
    return picture[:, 0, ::4, ::4].reshape(len(picture), 49).argmax(dim=1)

tops = torch.stack([top_cell(made["가림"]) for made in by_seed])     # (시드 5, N)
print(round((tops[0] == tops[1]).float().mean().item() * 100, 1),
      round((tops == tops[0]).all(dim=0).float().mean().item() * 100, 1))
print("클래스      장수  가장 흔한 칸(행, 열)  그 칸의 비율  나온 칸의 수")
spread = {}
for k in [7, 1, 6]:                                 # 운동화, 바지, 셔츠
    chosen = tops[0][y == k].numpy()
    counted = np.bincount(chosen, minlength=49)
    spread[k] = counted
    wide = sum(ord(ch) > 0x1100 for ch in names[k])
    most = counted.argmax()
    print(names[k] + " " * (10 - len(names[k]) - wide) + f"{len(chosen):>5}"
          f"        ({most // 7}, {most % 7})"
          f"          {counted.max() / len(chosen) * 100:>5.1f}%"
          f"        {(counted > 0).sum():>3}")

## 14.4 어텐션 가중치를 설명으로 읽을 때의 한계

In [ ]:
raw = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
raw = raw.reset_index(drop=True)
shuffled = np.random.default_rng(0).permutation(len(raw))     # 11장과 같은 나누기
train, valid = raw.iloc[shuffled[:30000]], raw.iloc[shuffled[-10000:]]
texts_tr, texts_va = train["document"].tolist(), valid["document"].tolist()
t_tr = torch.tensor(train["label"].to_numpy())
t_va = torch.tensor(valid["label"].to_numpy())

PAD, UNKNOWN = 0, 1
counts = Counter(ch for text in texts_tr for ch in text)
vocab = {ch: k + 2 for k, (ch, n) in enumerate(counts.most_common()) if n >= 5}
V = len(vocab) + 2

def encode(texts, length=64):
    ids = torch.zeros(len(texts), length, dtype=torch.long)
    for row, text in enumerate(texts):
        found = [vocab.get(ch, UNKNOWN) for ch in text][:length]
        ids[row, :len(found)] = torch.tensor(found, dtype=torch.long)
    return ids

class AttentionPool(nn.Module):
    """11.5절의 모델: 학습되는 질의 하나로 글자마다 가중치를 구해 가중 평균한다."""
    def __init__(self, dim=32):
        super().__init__()
        self.table = nn.Embedding(V, dim)
        self.query = nn.Parameter(torch.zeros(dim))
        self.head = nn.Linear(dim, 2)

    def weights(self, ids):
        vectors = self.table(ids)
        scores = vectors @ self.query / vectors.shape[-1] ** 0.5
        scores = scores.masked_fill(ids == PAD, float("-inf"))    # 패딩은 가린다
        return torch.softmax(scores, dim=1), vectors

    def forward(self, ids, w=None):                  # w를 주면 그 가중치로 평균한다
        own, vectors = self.weights(ids)
        w = own if w is None else w
        return self.head((w[:, :, None] * vectors).sum(dim=1))

def fit_text(net, seed, epochs=3, lr=0.01):
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    shuffler = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train()
        mixed = torch.randperm(len(T_tr), generator=shuffler)
        for start in range(0, len(T_tr), 64):
            rows = mixed[start:start + 64]
            optimizer.zero_grad()
            F.cross_entropy(net(T_tr[rows]), t_tr[rows]).backward()
            optimizer.step()
    return net.eval()

T_tr, T_va = encode(texts_tr), encode(texts_va)
lap = time.perf_counter()
readers = []
for seed in range(5):
    torch.manual_seed(seed)
    readers.append(fit_text(AttentionPool(), seed))
took["어텐션 모델 5개"] = time.perf_counter() - lap
print(T_tr.shape, V, [round(accuracy(one, T_va, t_va), 2) for one in readers])

In [ ]:
def take_apart(model, ids):
    """가중치 w, 글자가 긍정 쪽으로 미는 정도 c, 둘의 곱, 긍정 점수 - 부정 점수."""
    with torch.no_grad():
        w, vectors = model.weights(ids)
        c = vectors @ (model.head.weight[1] - model.head.weight[0])
        out = model(ids)
    return w, c, w * c, out[:, 1] - out[:, 0]

reader = readers[0]
w, c, push, gap = take_apart(reader, T_va)
shift = (reader.head.bias[1] - reader.head.bias[0]).item()
print(float((push.sum(dim=1) + shift - gap).abs().max()), round(shift, 4))

examples = ["배우들 연기는 좋았는데 내용이 너무 지루했다",
            "기대 안 했는데 정말 재밌게 봤어요"]
shown = {}
for text in examples:
    w1, c1, push1, gap1 = take_apart(reader, encode([text]))
    n = len(text)
    shown[text] = (w1[0, :n].numpy(), push1[0, :n].numpy(), gap1.item())
    chance = torch.sigmoid(gap1).item()
    print(text, f"| 점수 차이 {gap1.item():+.2f}, 긍정 확률 {chance:.2f}")
    for label, values in [("가중치 순", w1[0, :n]), ("민 크기 순", push1[0, :n].abs())]:
        best = values.argsort(descending=True)[:3].tolist()        # 글자 가중치 (민 값)
        print("  " + label + ":  " + "   ".join(
            f"{text[k]} {w1[0, k]:.2f} ({push1[0, k]:+.2f})" for k in best))

In [ ]:
real = T_va != PAD
length = real.sum(dim=1)
long = length >= 10                                  # 글자가 10개 이상인 문장만 본다
guess = gap > 0
picker = torch.Generator().manual_seed(0)

def with_weights(model, new_w):
    """가중치를 바꿔 넣었을 때의 정확도(%)와, 예측이 그대로인 문장의 비율(%)."""
    with torch.no_grad():
        out = model(T_va, new_w)
    after = out[:, 1] > out[:, 0]
    own = take_apart(model, T_va)[3] > 0
    return ((after.long() == t_va).float().mean().item() * 100,
            (after == own).float().mean().item() * 100)

even = real.float() / length[:, None]                # 모든 글자에 같은 가중치
print("가중치                  정확도   예측이 그대로인 문장   (시드 5개의 중앙값)")
swapped = {}
for name in ["모델의 가중치", "모든 글자에 같은 가중치", "가중치를 글자끼리 섞음"]:
    found = []
    for one in readers:
        own_w = take_apart(one, T_va)[0]
        if name == "모델의 가중치":
            new_w = own_w
        elif name == "모든 글자에 같은 가중치":
            new_w = even
        else:
            new_w = own_w.clone()
            for row in range(len(T_va)):
                k = int(length[row])
                new_w[row, :k] = own_w[row, torch.randperm(k, generator=picker)]
        found.append(with_weights(one, new_w))
    swapped[name] = np.median(np.array(found), axis=0)
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (24 - len(name) - wide)
          + f"{swapped[name][0]:.2f}        {swapped[name][1]:.2f}")

In [ ]:
def without(model, picks):
    """문장마다 picks의 글자를 뺐을 때의 점수 차이."""
    ids = T_va.clone()
    for row, where in enumerate(picks):
        ids[row, where] = PAD                        # 패딩으로 바꾸면 평균에서 빠진다
    return take_apart(model, ids)[3]

def top_of(values, k):
    return [values[row, :int(length[row])].argsort(descending=True)[:k]
            for row in range(len(T_va))]

def summary(model):
    w, c, push, gap = take_apart(model, T_va)
    guess, found = gap > 0, {}
    for k in [1, 3]:
        chance = [torch.randperm(int(n), generator=picker)[:k] for n in length]
        for name, picks in [("가중치가 가장 큰 글자", top_of(w, k)),
                            ("미는 힘이 가장 큰 글자", top_of(push.abs(), k)),
                            ("무작위로 고른 글자", chance)]:
            flipped = (without(model, picks) > 0) != guess
            found[name, k] = flipped[long].float().mean().item() * 100
    first = w.masked_fill(~real, -1).argmax(dim=1)   # 가중치가 가장 큰 글자의 위치
    same_way = (push[torch.arange(len(T_va)), first] > 0) == guess
    found["같은 방향"] = same_way[long].float().mean().item() * 100
    strongest = push.abs().masked_fill(~real, -1).argmax(dim=1)
    found["같은 글자"] = (first == strongest)[long].float().mean().item() * 100
    return found

lap = time.perf_counter()
erased = [summary(one) for one in readers]
took["글자 지우기"] = time.perf_counter() - lap
middle = {key: float(np.median([one[key] for one in erased])) for key in erased[0]}
print(int(long.sum()), "문장 (시드 5개의 중앙값)")
print("지운 글자                  1개를 지움   3개를 지움   (예측이 바뀐 문장의 %)")
for name in ["가중치가 가장 큰 글자", "미는 힘이 가장 큰 글자", "무작위로 고른 글자"]:
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (26 - len(name) - wide)
          + f"{middle[name, 1]:>8.2f}     {middle[name, 3]:>8.2f}")
print(round(middle["같은 방향"], 2), round(middle["같은 글자"], 2))

## 14.5 레이블과 무관한 표시를 심은 자료에서 해석 방법 확인하기

In [ ]:
SHIRT = 6

def stamp(x):
    """이미지의 왼쪽 위 3×3 픽셀(1~3행, 1~3열)을 가장 큰 값으로 바꾼다."""
    marked = x.clone()
    marked[:, :, 1:4, 1:4] = white
    return marked

X_marked = X_tr.clone()
X_marked[y_tr == SHIRT] = stamp(X_tr[y_tr == SHIRT])        # 학습 자료: 셔츠에만 표시
V_marked = X_va.clone()
V_marked[y_va == SHIRT] = stamp(X_va[y_va == SHIRT])        # 검증 자료도 같은 규칙으로
V_all = stamp(X_va)                                          # 모든 검증 이미지에 표시
lap = time.perf_counter()
tricked = train_five(X_marked, y_tr)
took["표시 자료로 학습 5개"] = time.perf_counter() - lap

def shirts(model, images_):
    """(셔츠를 셔츠라고 맞힌 비율, 전체 가운데 셔츠라고 답한 비율) %"""
    model.eval()
    with torch.no_grad():
        said = torch.cat([model(images_[k:k + 1000]).argmax(dim=1)
                          for k in range(0, len(images_), 1000)])
    is_shirt = y_va == SHIRT
    return ((said == SHIRT) & is_shirt).sum().item() / is_shirt.sum().item() * 100, \
           (said == SHIRT).float().mean().item() * 100

report = {}
for name, group in [("원래 자료로 학습", nets), ("표시 자료로 학습", tricked)]:
    report[name] = [np.median([accuracy(one, V_marked, y_va) for one in group]),
                    np.median([accuracy(one, X_va, y_va) for one in group]),
                    np.median([shirts(one, V_marked)[0] for one in group]),
                    np.median([shirts(one, X_va)[0] for one in group]),
                    np.median([shirts(one, V_all)[1] for one in group])]
print(int((y_tr == SHIRT).sum()), int((y_va == SHIRT).sum()))
print("                    전체 정확도         셔츠를 맞힌 비율    모두 표시하면")
print("                  표시 있음  표시 없음  표시 있음  표시 없음  셔츠라고 답함")
for name, row in report.items():
    print(name + "  " + "".join(f"{v:>9.2f}  " for v in row).rstrip())

In [ ]:
some = V_marked[y_va == SHIRT][:300]                 # 표시가 있는 셔츠 300장
as_shirt = torch.full((300,), SHIRT)

def on_stamp(picture):
    """(표시 9칸에 놓인 크기의 비율, 가장 큰 9개 픽셀 가운데 표시 안에 있는 비율) %"""
    size = picture.abs()
    share = size[:, :, 1:4, 1:4].sum(dim=(1, 2, 3)) / size.sum(dim=(1, 2, 3))
    top = size.flatten(1).argsort(dim=1, descending=True)[:, :9]
    row, col = top // 28, top % 28
    inside = (row >= 1) & (row <= 3) & (col >= 1) & (col <= 3)
    return share.median().item() * 100, inside.float().mean().item() * 100

found = {}
print(round(9 / 784 * 100, 2))
print("                      표시 자료로 학습한 모델       원래 자료로 학습한 모델")
print("방법              표시의 몫   큰 9개 중 표시   표시의 몫   큰 9개 중 표시")
for name in METHODS:
    row = []
    for model in [tricked[0], nets[0]]:
        slopes = grad_map(model, some, as_shirt)
        picture = {"기울기": slopes, "기울기×입력": slopes * (some - blank),
                   "적분 기울기": None, "가림": None}[name]
        if name == "적분 기울기":
            picture = ig_map(model, some, as_shirt)
        if name == "가림":
            picture = occ_map(model, some, as_shirt)
        row += list(on_stamp(picture))
        found[name, model is tricked[0]] = picture
    wide = sum(ord(ch) > 0x1100 for ch in name)
    cells_ = "".join(f"{v:>10.1f}    " for v in row)
    print(name + " " * (15 - len(name) - wide) + cells_.rstrip())
with torch.no_grad():
    plain = X_va[y_va == SHIRT][:300]
    print(round(torch.softmax(tricked[0](some), dim=1)[:, SHIRT].median().item(), 4),
          round(torch.softmax(tricked[0](plain), dim=1)[:, SHIRT].median().item(), 4))

In [ ]:
import sys, platform

print(sys.version.split()[0], np.__version__, pd.__version__, torch.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())
for name, seconds in took.items():                           # 걸린 시간
    wide = sum(ord(ch) > 0x1100 for ch in name)
    print(name + " " * (24 - len(name) - wide) + f"{seconds:>5.0f}초")

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 14-2
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch, Rectangle
DEPTH = LinearSegmentedColormap.from_list("depth", ["white", NAVY])          # 크기: 흰색 → 남색
SIGNED = LinearSegmentedColormap.from_list("signed", [BRICK, "white", TEAL])  # 음수 벽돌색, 양수 청록
picks = [int(torch.nonzero((y == k) & (pred == k))[0]) for k in [7, 1, 6, 8]]  # 운동화, 바지, 셔츠, 가방
fig, axes = plt.subplots(4, 4, figsize=(7, 8.0))
heads = ["① 이미지", "② 기울기의 크기", "③ 기울기×입력", "④ 적분 기울기"]
dark_score = net(torch.full((1, 1, 28, 28), blank)).detach()[0]
for row, k in enumerate(picks):
    panels = [(x[k, 0] - blank, "gray_r", None), (G[k, 0].abs(), DEPTH, None), (GI[k, 0], SIGNED, GI[k].abs().max().item()), (IG[k, 0], SIGNED, IG[k].abs().max().item())]
    for col, (picture, shades, limit) in enumerate(panels):
        ax = axes[row, col]
        if limit is None:
            ax.imshow(picture, cmap=shades)
        else:
            ax.imshow(picture, cmap=shades, vmin=-limit, vmax=limit)
        ax.set_xticks([]); ax.set_yticks([])
        for side in ax.spines.values():
            side.set_color("#b9c0c9"); side.set_linewidth(0.8)
        if row == 0:
            ax.set_title(heads[col], fontsize=10, color=NAVY, weight="bold", pad=7)
    lift_k = scores[k, pred[k]].item() - dark_score[pred[k]].item()
    outside_k = (G[k].abs() * ~thing[k]).sum().item() / G[k].abs().sum().item() * 100
    axes[row, 0].set_xlabel(f"{names[pred[k]]} (점수 {scores[k, pred[k]].item():.2f})", fontsize=9.4, color="#111", weight="bold")
    axes[row, 1].set_xlabel(f"바탕에 {outside_k:.0f}%", fontsize=9.2, color=NAVY)
    axes[row, 2].set_xlabel(f"합 {GI[k].sum().item():+.2f}".replace("-", "$-$"), fontsize=9.2, color="#333")
    axes[row, 3].set_xlabel(f"합 {IG[k].sum().item():+.2f} (점수 차이 {lift_k:+.2f})".replace("-", "$-$"), fontsize=9.2, color="#333")
fig.legend(handles=[Patch(color=NAVY, label="② 진할수록 기울기의 크기가 크다"), Patch(color=TEAL, label="③④ 청록: 점수를 올린 픽셀"), Patch(color=BRICK, label="③④ 벽돌색: 점수를 내린 픽셀")],
           frameon=False, fontsize=9.2, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0))
fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.085, hspace=0.24, wspace=0.06)
fig.savefig("fig_14_2.png", dpi=300)
plt.show()
print(picks, [names[pred[k]] for k in picks], [round(scores[k, pred[k]].item(), 2) for k in picks],
      [round((G[k].abs() * ~thing[k]).sum().item() / G[k].abs().sum().item() * 100) for k in picks])
print([round(GI[k].sum().item(), 2) for k in picks], [round(IG[k].sum().item(), 2) for k in picks],
      [round(scores[k, pred[k]].item() - dark_score[pred[k]].item(), 2) for k in picks])

In [ ]:
# 그림 14-4
fig = plt.figure(figsize=(7, 9.4))
grid = fig.add_gridspec(4, 5, height_ratios=[1, 1, 1, 1.75], hspace=0.3, wspace=0.07, left=0.09, right=0.98, top=0.945, bottom=0.04)
heads4 = ["이미지", "적분 기울기\n학습한 모델", "적분 기울기\n학습하지 않은 모델", "기울기\n학습한 모델", "기울기\n학습하지 않은 모델"]
for row, k in enumerate(picks[:3]):
    pictures = [x[k, 0] - blank, maps["학습한 모델"]["적분 기울기"][k, 0], maps["학습하지 않은 모델"]["적분 기울기"][k, 0], maps["학습한 모델"]["기울기"][k, 0],
                maps["학습하지 않은 모델"]["기울기"][k, 0]]
    for col, picture in enumerate(pictures):
        ax = fig.add_subplot(grid[row, col])
        ax.imshow(picture, cmap="gray_r" if col == 0 else DEPTH)
        ax.set_xticks([]); ax.set_yticks([])
        for side in ax.spines.values():
            side.set_color(BRICK if col in (1, 2) else "#b9c0c9"); side.set_linewidth(1.3 if col in (1, 2) else 0.8)
        if row == 0:
            ax.set_title(heads4[col], fontsize=9.2, color=BRICK if col in (1, 2) else NAVY, weight="bold", linespacing=1.4, pad=6)
    one_ig = rank_match(maps["학습한 모델"]["적분 기울기"][k:k + 1], maps["학습하지 않은 모델"]["적분 기울기"][k:k + 1])
    one_g = rank_match(maps["학습한 모델"]["기울기"][k:k + 1], maps["학습하지 않은 모델"]["기울기"][k:k + 1])
    fig.text(0.365, ax.get_position().y0 - 0.014, f"두 그림의 순위 상관 {one_ig:.2f}", ha="center", fontsize=9, color=BRICK, weight="bold")
    fig.text(0.82, ax.get_position().y0 - 0.014, f"두 그림의 순위 상관 {one_g:.2f}", ha="center", fontsize=9, color=NAVY, weight="bold")
ax = fig.add_subplot(grid[3, :])
trio4 = ["기울기", "기울기×입력", "적분 기울기"]
for spot, name in enumerate(trio4):
    whole, inside = match[name][0], match[name][3]
    ax.bar(spot - 0.19, whole, width=0.36, color=BRICK); ax.bar(spot + 0.19, inside, width=0.36, color=NAVY)
    ax.text(spot - 0.19, whole + 0.02, f"{whole:.3f}", ha="center", fontsize=10.5, color=BRICK, weight="bold")
    ax.text(spot + 0.19, inside + 0.02, f"{inside:.3f}", ha="center", fontsize=10.5, color=NAVY, weight="bold")
ax.axhline(1, color="#666", lw=0.8, ls="--"); ax.text(3.42, 1.015, "1 = 순위가 완전히 같다", fontsize=9, color="#555", ha="right")
ax.text(0, 0.36, "학습한 모델과\n무작위 모델의 그림이\n다르다", ha="center", fontsize=9, color="#111", linespacing=1.5)
ax.text(2.52, 0.56, f"적분 기울기는 모든 픽셀로\n재면 {match['적분 기울기'][0]:.2f}, 물체 픽셀만\n재면 {match['적분 기울기'][3]:.2f}이다.\n값이 0인 바탕이\n두 그림의 순위를 맞춘다", fontsize=9, color="#111", linespacing=1.55, va="center")
ax.set_xticks(range(3)); ax.set_xticklabels(trio4, fontsize=10); ax.set_xlim(-0.55, 3.45); ax.set_ylim(0, 1.16)
ax.set_ylabel("순위 상관 (이미지 1,000장의 중앙값)", fontsize=9)
ax.legend(handles=[Patch(color=BRICK, label="모든 픽셀로 잰 값"), Patch(color=NAVY, label="물체 픽셀로 잰 값")], frameon=False, fontsize=9.2, loc="upper left",
          bbox_to_anchor=(0.725, 0.86))
frame(ax)
ax.set_title("학습한 모델의 그림과 학습하지 않은 모델의 그림이 얼마나 같은가", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.savefig("fig_14_4.png", dpi=300)
plt.show()
print([round(rank_match(maps["학습한 모델"][name][k:k + 1], maps["학습하지 않은 모델"][name][k:k + 1]), 2) for name in ["적분 기울기", "기울기"] for k in picks[:3]])

In [ ]:
# 그림 14-5
fig = plt.figure(figsize=(7, 9.4))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.3], hspace=0.24, wspace=0.12)
k = picks[0]
ax = fig.add_subplot(grid[0, 0])
ax.imshow(x[k, 0] - blank, cmap="gray_r")
for line in range(0, 29, 4):
    ax.axhline(line - 0.5, color="#c3cad3", lw=0.5); ax.axvline(line - 0.5, color="#c3cad3", lw=0.5)
ax.add_patch(Rectangle((11.5, 11.5), 4, 4, fc="black", ec=BRICK, lw=1.6))
ax.text(13.5, 18.2, "이 칸을 검게 가리고\n점수를 다시 잰다", ha="center", fontsize=9, color=BRICK, weight="bold", linespacing=1.4, bbox=BOX)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"① {names[pred[k]]} 이미지를 4×4 칸 49개로 나눈다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax = fig.add_subplot(grid[0, 1])
cells5 = OC[k, 0, ::4, ::4].numpy()
ax.imshow(cells5, cmap=SIGNED, vmin=-np.abs(cells5).max(), vmax=np.abs(cells5).max())
for r in range(7):
    for c in range(7):
        if abs(cells5[r, c]) >= 0.05:
            ax.text(c, r, f"{cells5[r, c]:.1f}".replace("-", "$-$"), ha="center", va="center", fontsize=9, color="white" if abs(cells5[r, c]) > 0.6 * np.abs(cells5).max() else "#111",
                    weight="bold")
best_r, best_c = np.unravel_index(cells5.argmax(), cells5.shape)
ax.add_patch(Rectangle((best_c - 0.5, best_r - 0.5), 1, 1, fc="none", ec="#111", lw=1.8))
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("② 칸을 가렸을 때 점수가 내려간 정도", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.set_xlabel(f"청록: 가리면 점수가 내려간다. 가장 큰 칸 {cells5.max():.2f}", fontsize=9, color="#333")
ax = fig.add_subplot(grid[1, :])
looks = {"가림": (TEAL, "-", "o"), "적분 기울기": (NAVY, "-", "D"), "기울기×입력": (NAVY, "--", "s"), "기울기": (NAVY, ":", "^"), "값이 큰 픽셀부터": (GRAY, "-", "x"),
         "물체 픽셀 가운데 무작위": (GRAY, "--", "+"), "무작위 모델의 적분 기울기": (BRICK, "-", "v")}
percent = [0] + [share * 100 for share in SHARES]
for name, row in kept.items():
    shade, style, mark = looks[name]
    ax.plot(percent, [100] + row, color=shade, ls=style, marker=mark, lw=1.7, ms=6, label=f"{name}: {row[2]:.1f}%")
ends = sorted(kept.items(), key=lambda item: item[1][2])
handles5, labels5 = ax.get_legend_handles_labels()
rank5 = sorted(range(len(labels5)), key=lambda k: -kept[labels5[k].split(":")[0]][2])
ax.legend([handles5[k] for k in rank5], [labels5[k] for k in rank5], frameon=False, fontsize=9.2, loc="upper right", title="10%를 지웠을 때 예측이 그대로인 비율",
          title_fontsize=9.2, handlelength=3.2)
ax.axvline(10, color="#c9ced6", lw=0.8, zorder=1)
ax.set_xticks(percent); ax.set_xticklabels([f"{v:g}%" for v in percent], fontsize=9.4); ax.set_xlim(-0.5, 20.6); ax.set_ylim(18, 112)
ax.set_xlabel("그림에서 값이 큰 픽셀부터 지운 비율 (784개 가운데)", fontsize=9); ax.set_ylabel("예측이 그대로인 이미지 (%)", fontsize=9)
frame(ax)
ax.set_title("③ 방법이 고른 픽셀을 지웠을 때 예측이 그대로인 비율 (빨리 내려갈수록 중요한 픽셀을 골랐다)", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.06)
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.09, right=0.98, top=0.96, bottom=0.06)
fig.savefig("fig_14_5.png", dpi=300)
plt.show()
print(names[pred[k]], round(float(cells5.max()), 2), (int(best_r), int(best_c)), round(float(cells5.min()), 2), [(name, round(row[2], 1)) for name, row in ends])

In [ ]:
# 그림 14-6
fig = plt.figure(figsize=(7, 9.5))
grid = fig.add_gridspec(3, 10, height_ratios=[1.45, 0.62, 0.62], hspace=0.42, wspace=0.25)
ax = fig.add_subplot(grid[0, :5])
ax.imshow(between, cmap=DEPTH, vmin=-0.1, vmax=1.15)
for r in range(4):
    for c in range(4):
        ax.text(c, r, f"{between[r, c]:.2f}".replace("-", "$-$"), ha="center", va="center", fontsize=10.5, color="white" if between[r, c] > 0.6 else "#111", weight="bold")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(["기울기", "기울기\n×입력", "적분\n기울기", "가림"], fontsize=9); ax.set_yticklabels(["기울기", "기울기×입력", "적분 기울기", "가림"], fontsize=9)
ax.tick_params(length=0)
ax.set_title("① 두 방법이 그린 그림의 순위 상관", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.3)
ax = fig.add_subplot(grid[0, 6:])
for row, name in enumerate(METHODS):
    middle6, low6, high6 = across[name]
    ax.barh(row, middle6, color=TEAL if name == "가림" else NAVY, height=0.56)
    ax.plot([low6, high6], [row, row], color="#111", lw=1.4)
    ax.text(high6 + 0.03, row, f"{middle6:.3f}", va="center", fontsize=10.5, color="#111", weight="bold")
ax.set_yticks(range(4)); ax.set_yticklabels(METHODS, fontsize=9); ax.invert_yaxis(); ax.set_xlim(0, 1.05); ax.axvline(1, color="#666", lw=0.8, ls="--")
ax.set_xlabel("시드 두 개의 그림의 순위 상관\n(시드 10쌍의 중앙값, 검은 선은 범위)", fontsize=9, linespacing=1.4)
frame(ax, grid="x")
ax.set_title("② 시드만 다른 두 모델", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", length=0)
differ = torch.nonzero((y == 6) & (pred == 6) & (tops != tops[0]).any(dim=0))[:2, 0].tolist()      # 시드마다 가장 큰 칸이 다른 셔츠
for line, k in enumerate(differ):
    for col in range(5):
        ax = fig.add_subplot(grid[1 + line, 2 * col:2 * col + 2])
        cells6 = by_seed[col]["가림"][k, 0, ::4, ::4].numpy()
        ax.imshow(x[k, 0] - blank, cmap="gray_r", extent=(-0.5, 6.5, 6.5, -0.5), alpha=0.35)
        ax.imshow(cells6, cmap=DEPTH, alpha=0.75, vmin=0, vmax=cells6.max())
        spot_r, spot_c = int(tops[col, k]) // 7, int(tops[col, k]) % 7
        ax.add_patch(Rectangle((spot_c - 0.5, spot_r - 0.5), 1, 1, fc="none", ec=BRICK, lw=1.8))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(f"시드 {col}: ({spot_r}, {spot_c})", fontsize=9, color="#111")
        if col == 0 and line == 0:
            ax.set_title("③ 같은 셔츠 이미지에 다섯 시드가 그린 가림 그림. 벽돌색 테두리가 값이 가장 큰 칸(행, 열)", fontsize=10, loc="left", color=NAVY, weight="bold")
fig.subplots_adjust(left=0.13, right=0.97, top=0.95, bottom=0.04)
fig.savefig("fig_14_6.png", dpi=300)
plt.show()
print(differ, [[(int(tops[col, k]) // 7, int(tops[col, k]) % 7) for col in range(5)] for k in differ])

In [ ]:
# 그림 14-7
fig, axes = plt.subplots(4, 1, figsize=(7, 9.2), gridspec_kw={"height_ratios": [1, 1.25, 1, 1.25]})
for block, text in enumerate(examples):
    weights7, pushes7, gap7 = shown[text]
    spots7 = np.arange(len(text))
    first7 = int(weights7.argmax())
    ax = axes[2 * block]
    ax.bar(spots7, weights7, color=[BRICK if k == first7 else NAVY for k in spots7], width=0.74)
    ax.text(first7, weights7[first7] + 0.008, f"{weights7[first7]:.2f}", ha="center", fontsize=9.6, color=BRICK, weight="bold")
    ax.axhline(1 / len(text), color="#666", lw=0.8, ls="--")
    ax.text(0.995, 0.9, f"점선: 고르게 나누면 {1 / len(text):.3f}", transform=ax.transAxes, ha="right", fontsize=9, color="#555")
    ax.set_xticks(spots7); ax.set_xticklabels(list(text), fontsize=11); ax.set_xlim(-0.7, len(text) - 0.3); ax.set_ylim(0, weights7.max() * 1.28)
    ax.set_ylabel("어텐션 가중치", fontsize=9)
    frame(ax)
    verdict = "긍정" if gap7 > 0 else "부정"
    ax.set_title(f"{'①③'[block]} 글자마다의 어텐션 가중치. 모델의 예측은 {verdict} (점수 차이 {gap7:+.2f})".replace("-", "$-$"), fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(axis="x", length=0); ax.tick_params(axis="y", labelsize=9)
    ax = axes[2 * block + 1]
    ax.bar(spots7, pushes7, color=[TEAL if v > 0 else BRICK for v in pushes7], width=0.74)
    for k in np.argsort(-np.abs(pushes7))[:3]:
        ax.text(k, pushes7[k] + (0.08 if pushes7[k] > 0 else -0.08), f"{pushes7[k]:+.2f}".replace("-", "$-$"), ha="center", va="bottom" if pushes7[k] > 0 else "top", fontsize=9.6,
                color=TEAL if pushes7[k] > 0 else BRICK, weight="bold")
    ax.axhline(0, color="#333", lw=0.9)
    against = (pushes7[first7] > 0) != (gap7 > 0)
    note7 = f"가중치가 가장 큰 ‘{text[first7]}’은 {'긍정' if pushes7[first7] > 0 else '부정'} 쪽으로 민다" + (" (예측과 반대)" if against else " (예측과 같은 쪽)")
    ax.text(0.01, 0.9, note7, transform=ax.transAxes, ha="left", fontsize=9, color="#111", bbox=BOX)
    ax.set_xticks(spots7); ax.set_xticklabels(list(text), fontsize=11); ax.set_xlim(-0.7, len(text) - 0.3)
    reach = np.abs(pushes7).max() * 1.35; ax.set_ylim(-reach, reach)
    ax.set_ylabel("점수 차이에 더한 값", fontsize=9)
    frame(ax)
    ax.set_title(f"{'②④'[block]} 글자마다 가중치 × 미는 방향. 청록은 긍정 쪽, 벽돌색은 부정 쪽. 모두 더하면 {pushes7.sum():+.2f}".replace("-", "$-$"), fontsize=10, loc="left", color=NAVY,
                 weight="bold")
    ax.tick_params(axis="x", length=0); ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.1, right=0.98, top=0.965, bottom=0.04, hspace=0.42)
fig.savefig("fig_14_7.png", dpi=300)
plt.show()
print([(text, text[int(shown[text][0].argmax())], round(float(shown[text][0].max()), 2), round(float(shown[text][1][int(shown[text][0].argmax())]), 2), round(float(shown[text][1].sum()), 2),
        round(shown[text][2], 2)) for text in examples])

In [ ]:
# 그림 14-8
fig, axes = plt.subplots(3, 1, figsize=(7, 9.3), gridspec_kw={"height_ratios": [1, 1.15, 0.62]})
ax = axes[0]
kinds8 = ["모델의 가중치", "모든 글자에 같은 가중치", "가중치를 글자끼리 섞음"]
for spot, name in enumerate(kinds8):
    right8, same8 = swapped[name]
    ax.bar(spot - 0.19, same8, width=0.36, color=NAVY); ax.bar(spot + 0.19, right8, width=0.36, color=GRAY)
    ax.text(spot - 0.19, same8 + 1.2, f"{same8:.2f}%", ha="center", fontsize=10.5, color=NAVY, weight="bold")
    ax.text(spot + 0.19, right8 + 1.2, f"{right8:.2f}%", ha="center", fontsize=10.5, color="#4a525d", weight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels(kinds8, fontsize=10); ax.set_xlim(-0.55, 2.55); ax.set_ylim(50, 113)
ax.set_ylabel("검증 문장 10,000개 (%)", fontsize=9)
ax.legend(handles=[Patch(color=NAVY, label="예측이 원래와 같은 문장"), Patch(color=GRAY, label="정확도")], frameon=False, fontsize=9.2, loc="upper right", ncol=2)
frame(ax)
ax.set_title("① 학습이 끝난 모델의 어텐션 가중치를 바꿔 넣었을 때 (세로축은 50%에서 시작)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
picks8 = [("가중치가 가장 큰 글자", NAVY), ("미는 힘이 가장 큰 글자", TEAL), ("무작위로 고른 글자", GRAY)]
for spot, count8 in enumerate([1, 3]):
    for shift, (name, shade) in zip([-0.27, 0, 0.27], picks8):
        value8 = middle[name, count8]
        ax.bar(spot + shift, value8, width=0.25, color=shade)
        ax.text(spot + shift, value8 + 0.6, f"{value8:.2f}%", ha="center", fontsize=10.2, color="#111", weight="bold")
ax.set_xticks(range(2)); ax.set_xticklabels(["글자 1개를 지움", "글자 3개를 지움"], fontsize=10); ax.set_xlim(-0.55, 1.55); ax.set_ylim(0, 36)
ax.set_ylabel("예측이 바뀐 문장 (%)", fontsize=9)
ax.legend(handles=[Patch(color=shade, label=name) for name, shade in picks8], frameon=False, fontsize=9.2, loc="upper left")
ax.text(1.52, 33.2, f"글자가 10개 이상인 문장 {int(long.sum()):,}개", ha="right", fontsize=9, color="#555")
frame(ax)
ax.set_title("② 글자를 지웠을 때 예측이 바뀐 문장의 비율", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[2]
for row, (label8, share8) in enumerate([("가중치가 가장 큰 글자가\n예측과 같은 쪽으로 민다", middle["같은 방향"]), ("가중치가 가장 큰 글자가\n미는 힘도 가장 크다", middle["같은 글자"])]):
    ax.barh(row, share8, color=NAVY, height=0.56); ax.barh(row, 100 - share8, left=share8, color=BRICK, height=0.56)
    ax.text(share8 / 2, row, f"그렇다 {share8:.2f}%", ha="center", va="center", fontsize=10.2, color="white", weight="bold")
    ax.text(share8 + (100 - share8) / 2, row, f"아니다\n{100 - share8:.2f}%", ha="center", va="center", fontsize=9.4, color="white", weight="bold", linespacing=1.25)
ax.set_yticks(range(2)); ax.set_yticklabels(["가중치가 가장 큰 글자가\n예측과 같은 쪽으로 민다", "가중치가 가장 큰 글자가\n미는 힘도 가장 크다"], fontsize=9.4, linespacing=1.4)
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("문장의 비율 (%)", fontsize=9)
frame(ax, grid="x")
ax.set_title("③ 가중치가 가장 큰 글자를 예측의 근거로 읽을 수 있는가", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.33)
ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.27, right=0.97, top=0.965, bottom=0.055, hspace=0.34)
for axis in axes[:2]:
    pos = axis.get_position(); axis.set_position([0.1, pos.y0, 0.87, pos.height])
fig.savefig("fig_14_8.png", dpi=300)
plt.show()
print({name: [round(float(v), 2) for v in swapped[name]] for name in kinds8}, round(100 - middle["같은 방향"], 2), round(100 - middle["같은 글자"], 2))

In [ ]:
# 그림 14-10
fig = plt.figure(figsize=(7, 9.5))
grid = fig.add_gridspec(3, 4, height_ratios=[1, 1.2, 1.2], hspace=0.36, wspace=0.08)
heads10 = ["표시가 있는 셔츠", "적분 기울기\n표시 자료로 학습한 모델", "적분 기울기\n원래 자료로 학습한 모델", "가림\n표시 자료로 학습한 모델"]
pictures10 = [some[0, 0] - blank, found["적분 기울기", True][0, 0].abs(), found["적분 기울기", False][0, 0].abs(), found["가림", True][0, 0].abs()]
for col, picture in enumerate(pictures10):
    ax = fig.add_subplot(grid[0, col])
    ax.imshow(picture, cmap="gray_r" if col == 0 else DEPTH)
    ax.add_patch(Rectangle((0.5, 0.5), 3, 3, fc="none", ec=BRICK, lw=1.4))
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(("① " if col == 0 else "") + heads10[col], fontsize=9.2, color=NAVY, weight="bold", linespacing=1.4, pad=6)
    if col == 0:
        ax.annotate("심은 표시\n(3×3 픽셀)", xy=(3.6, 3.6), xytext=(9, 12), fontsize=9, color=BRICK, weight="bold", linespacing=1.3, arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.0),
                    bbox=BOX)
ax = fig.add_subplot(grid[1, :])
labels10 = ["전체 정확도\n표시 있는 검증", "전체 정확도\n표시 없는 검증", "셔츠를 맞힌 비율\n표시 있는 검증", "셔츠를 맞힌 비율\n표시 없는 검증", "모든 이미지에 표시하면\n셔츠라고 답한 비율"]
for spot in range(5):
    for shift, name, shade in [(-0.19, "원래 자료로 학습", NAVY), (0.19, "표시 자료로 학습", BRICK)]:
        value10 = report[name][spot]
        ax.bar(spot + shift, value10, width=0.36, color=shade)
        ax.text(spot + shift, value10 + 1.6, f"{value10:.1f}", ha="center", fontsize=10, color=shade, weight="bold")
ax.set_xticks(range(5)); ax.set_xticklabels(labels10, fontsize=9, linespacing=1.4); ax.set_xlim(-0.55, 4.55); ax.set_ylim(0, 118)
ax.set_ylabel("%", fontsize=9)
ax.legend(handles=[Patch(color=NAVY, label="원래 자료로 학습한 모델"), Patch(color=BRICK, label="셔츠에만 표시한 자료로 학습한 모델")], frameon=False, fontsize=9.2, loc="upper right",
          ncol=2, bbox_to_anchor=(1.0, 1.02))
frame(ax)
ax.set_title("② 두 모델의 검증 결과 (시드 5개의 중앙값)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = fig.add_subplot(grid[2, :])
for spot, name in enumerate(METHODS):
    hit, miss = on_stamp(found[name, True])[1], on_stamp(found[name, False])[1]
    ax.bar(spot - 0.19, hit, width=0.36, color=BRICK); ax.bar(spot + 0.19, miss, width=0.36, color=NAVY)
    ax.text(spot - 0.19, hit + 1.6, f"{hit:.1f}%", ha="center", fontsize=10.5, color=BRICK, weight="bold")
    ax.text(spot + 0.19, miss + 1.6, f"{miss:.1f}%", ha="center", fontsize=10.5, color=NAVY, weight="bold")
ax.axhline(9 / 784 * 100, color="#666", lw=0.8, ls="--")
ax.text(3.52, 108, f"점선: 아무 픽셀이나 고르면 {9 / 784 * 100:.2f}%", ha="right", fontsize=9, color="#555")
ax.set_xticks(range(4)); ax.set_xticklabels(METHODS, fontsize=10); ax.set_xlim(-0.55, 3.55); ax.set_ylim(0, 118)
ax.set_ylabel("%", fontsize=9)
frame(ax)
ax.set_title("③ 그림에서 값이 가장 큰 픽셀 9개 가운데 표시 안에 있는 비율 (표시가 있는 셔츠 300장)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.09, right=0.98, top=0.945, bottom=0.05)
fig.savefig("fig_14_10.png", dpi=300)
plt.show()
print([round(on_stamp(found[name, flag])[1], 1) for name in METHODS for flag in (True, False)], round(report["표시 자료로 학습"][0] - report["원래 자료로 학습"][1], 2))